# ARE SD - CLASSIFICATION DIAGNOSTIQUE D'ARBRES

> RAYNAL COBO Alexa, Vos Noms et Prenoms

**Importation de données et bibliothèques à utiliser**

In [152]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import random
import math
import time
import itertools
random.seed(0)


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/are-sd-2025-classification-diagnostique-d-arbres/benchmark.csv
/kaggle/input/are-sd-2025-classification-diagnostique-d-arbres/prev.csv
/kaggle/input/are-sd-2025-classification-diagnostique-d-arbres/train.csv


# Données à manipuler

In [153]:
data = pd.read_csv('/kaggle/input/are-sd-2025-classification-diagnostique-d-arbres/train.csv', index_col = 'ID_ARBRE')
prev_data = pd.read_csv('/kaggle/input/are-sd-2025-classification-diagnostique-d-arbres/prev.csv', index_col = 'ID_ARBRE')
data.head()

,quartier,site,genre_arbre,situation,type_sol,surf_permeable,classe_age,hauteur,classe_hauteur,diametre,...,fissure_houppier,ecorce_incluse_houppier,bois_mort_houppier,plaie_houppier,observation_houppier,esperance_maintien,contrainte,classification_diagnostic,Long,Lat
ID_ARBRE,,,,,,,,,,,,,,,,,,,,,
0,Quartier 3 - Lycée International,RN13,Tilia,Alignement,MA,4.0,A,500,H1,31.830989,...,HPF,Non,HPBM,HPLS,0,1.0,Non,C1,2.064,48.900
1,Quartier 2 - Alsace - Pereire,Avenue du Maréchal Foch,Tilia,Alignement,MA,4.0,A,800,H2,76.394373,...,HFO,Non,HPBM,HPLNC,Fissure ouverte axe2 2M Sud Est,2.0,Oui,C4,2.085,48.903
2,Quartier 2 - Alsace - Pereire,RN13,Tilia,Alignement,GR,4.0,A,500,H1,38.197186,...,HPF,Non,HPBM,HPLS,0,1.0,Non,C1,2.070,48.899
3,Quartier 1 - Cœur de Ville et Quatier forestier,Avenue Gambetta,Tilia,Alignement,Gr,1.0,A,600,H2,66.845076,...,HPF,Non,HPBM,HPLS,Branches cassées,1.0,Non,C2,2.099,48.897
4,Quartier 1 - Cœur de Ville et Quatier forestier,Rue Thiers,Cercis,Isolé,MA,20.0,JA,400,H1,25.464791,...,HPF,Non,HPBM,HPPL,0,1.0,Non,C2,2.101,48.898


Dans cette première partie, nous nous intéressons à rendre les données utilisables, en définissant des fonctions telles que label_encoder et normalise, qui permettent de convertir des chaînes de caractères en entiers et de normaliser nos données (ce qui est particulièrement utile pour l'implémentation d'algorithmes comme KNN).

# Fonctions de pretraitement des donnees

In [154]:
def label_encoder(feature_vec):
    """
    Encode les étiquettes extraites d'une matrice de données d'entraînement.
    L'encodage consiste à remplacer les labels (chaînes de caractères) par des entiers 
    afin de pouvoir calculer une distance entre deux données, ce qui est essentiel pour 
    certains algorithmes comme KNN.

    Arguments:
        feature_vec : numpy array ou liste
            Un vecteur contenant les labels à encoder.

    Retourne:
        - encoded_labels : numpy array
            Un vecteur des labels encodés sous forme d'entiers.
        - unique_values : numpy array
            Les valeurs uniques initiales correspondant aux labels originaux.
    """
    unique_values, encoded_labels = np.unique(feature_vec, return_inverse=True)
    # Retourne les indices des modalités tout en préservant la forme originale (par exemple, si c'était une matrice, elle reste une matrice)
    # Retourne aussi le mapping pour pouvoir retrouver les valeurs originales.
    return encoded_labels.reshape(feature_vec.shape), unique_values 


# Fonction inverse pour décoder les labels
def decode_labels(encoded_vec, mapping):
    """
    Décode des étiquettes encodées en utilisant un mapping fourni.
    
    Arguments:
        encoded_vec : numpy array
            Un vecteur des labels encodés sous forme d'entiers.
        mapping : numpy array
            Les valeurs uniques originales correspondant aux labels encodés.

    Retourne:
        numpy array
            Un vecteur des labels décodés sous forme de chaînes de caractères.
    """
    return mapping[np.array(encoded_vec)]

# Cette fonction est principalement utilisée pour KNN
def normalise(X_train):
    """
    Normalise les données en appliquant la formule de la centrée-réduite.
    Cette normalisation transforme les données de façon à ce qu'elles aient une moyenne 
    de 0 et une variance de 1.

    Arguments:
        X_train : numpy array
            Une matrice avec n paramètres (colonnes) et m lignes de données à normaliser.

    Retourne:
        numpy array
            La matrice normalisée avec les mêmes dimensions que X_train.
    """
    return (X_train - np.mean(X_train, axis=0)) / np.var(X_train, axis=0)

> Démontrons le fonctionnement:

In [155]:
A = np.array(["C1", "C2", "C3"])
vector, mapping = label_encoder(A)
print(vector, mapping)

[0 1 2] ['C1' 'C2' 'C3']


In [156]:
decode_labels(*label_encoder(A)) #On verifie bien que ce sont des fonctions inverses

array(['C1', 'C2', 'C3'], dtype='<U2')

In [157]:
B = np.array([2, 1, 0])
decode_labels(B, mapping)

array(['C3', 'C2', 'C1'], dtype='<U2')

# Implémentation des modèles d'apprentissage


Ici, nous allons montrer l'implémentation de chacun de nos modèles (KNN et Random Forest), pour finir avec un modèle ensembliste. L'ordre d'implémentation de chaque modèle est le suivant :

1. **Développement des fonctions pour le modèle**
2. **Raffinement des hyperparamètres**
3. **Exécution de l'algorithme avec les hyperparamètres affinés**

Finalement, nous présenterons le modèle final qui combine les prédictions des deux modèles via un **vote ensembliste**.


# KNN

Definition de la fonction distance euclidienne:

In [158]:
def euclidian_distance(x_train, x_test):
    """
    Calcule la distance euclidienne entre x_train et x_test.

    La distance entre une donnée d'entraînement et une donnée test est défine comme la racine carrée de :
                sum( (x_train[i] - x_test[i])^2 )
                 i
    Cette somme peut être exprimée sous forme vectorielle :

    sum( (x_train[i] - x_test[i])^2 ) = < x_train - x_test | x_train - x_test> 
     i
                                      = <x_train | x_train> + <x_test | x_test> - 2.<x_train | x_test>
    """
    sqr_dist = -2*x_test@x_train.T + np.sum(x_train**2,axis=1) + np.sum(x_test**2,axis=1)[:,np.newaxis]
    sqr_dist[sqr_dist<0] = 0.
    # Retour de la distance euclidienne
    return np.sqrt(sqr_dist)

In [159]:
# X_train est une matrice avec n colonnes représentant les paramètres pris en compte pour la classification,
# et m lignes qui représentent le nombre d'exemples de données d'entraînement.
# Y_train est un vecteur contenant les classes des données d'entraînement.
# X_test est une matrice similaire à X_train, mais sans les vecteurs de réponse associés (c'est-à-dire les classes que l'on veut prédire).
# k est le nombre de voisins à prendre en compte lors de la classification.

def knn(k, X_train, X_test, Y_train):
    """
    Effectue la classification des exemples de X_test en utilisant l'algorithme K-Nearest Neighbors (KNN).
    On suppose que les variables de X_train et X_test sont discrétisées et/ou normalisées.
    
    Arguments :
    - k : Nombre de voisins à considérer pour la classification.
    - X_train : Matrice des données d'entraînement, de forme (m, n) où m est le nombre d'exemples
      et n est le nombre de caractéristiques.
    - X_test : Matrice des données de test, de forme (p, n), où p est le nombre d'exemples de test.
    - Y_train : Vecteur contenant les classes des exemples d'entraînement, de forme (m,).

    Retourne :
    - Un vecteur contenant les classes prédites pour chaque exemple de X_test, de forme (p,).
    """
    # Calcul de la distance euclidienne entre chaque exemple de X_test et tous les exemples de X_train
    dist = euclidian_distance(X_train, X_test)  # Renvoie une matrice de distances euclidiennes
    # Shape de dist : (len(X_test), len(X_train))

    # Trouver les indices des k voisins les plus proches pour chaque exemple de X_test
    nearest_neighbors = np.argsort(dist, axis=1)[:, :k]  # Shape : (len(X_test), k)
    
    # Obtenir les classes des k voisins les plus proches pour chaque exemple de X_test
    k_nearest_labels = Y_train[nearest_neighbors]
    
    # Pour chaque exemple de X_test, obtenir la classe majoritaire parmi les k voisins
    classe = np.array([np.bincount(labels).argmax() for labels in k_nearest_labels])
    
    return classe


> Jusqu'ici, nous avons un algorithme fonctionnel qui permet de créer une prédiction avec KNN. Cependant, pour trouver le meilleur k et la meilleure combinaison de paramètres, nous avons besoin d'une métrique non biaisée qui évalue le taux d'erreur en utilisant la validation croisée. De cette manière, nous évitons les problèmes de surapprentissage.

In [160]:
def knn_error(k, X_train, X_test, Y_train, Y_test):
    """
    Calcule le taux d'erreur de la prédiction de l'algorithme KNN sur l'ensemble de test.
    
    Arguments :
    - k : Nombre de voisins à considérer pour la classification.
    - X_train : Matrice des données d'entraînement, de forme (m, n), où m est le nombre d'exemples et n est le nombre de caractéristiques.
    - X_test : Matrice des données de test, de forme (p, n), où p est le nombre d'exemples de test.
    - Y_train : Vecteur contenant les classes des exemples d'entraînement, de forme (m,).
    - Y_test : Vecteur contenant les classes réelles des exemples de test, de forme (p,).
    
    Retourne :
    - Le taux d'erreur, qui est la proportion des prédictions incorrectes.
    """
    # Prédire les classes pour X_test avec l'algorithme KNN
    Y_pred = knn(k, X_train, X_test, Y_train)
    
    # Calculer le taux d'erreur
    error_rate = np.mean(Y_test != Y_pred)
    
    return error_rate

In [161]:
def k_fold_split(X, Y, k=10, seed=42):
    """
    Effectue une division des données en k parties égales pour effectuer la validation croisée (k-fold cross-validation).
    
    Paramètres :
    - X : Matrice des données (numpy array), de forme (n_samples, n_features).
    - Y : Vecteur des classes (numpy array), de forme (n_samples,).
    - k : Nombre de partitions (folds) pour la validation croisée (par défaut k=10).
    - seed : Valeur pour initialiser le générateur de nombres aléatoires, permettant de reproduire les résultats (par défaut seed=42).
    
    Retourne :
    - Une liste de tuples (X_train, X_test, Y_train, Y_test) pour chaque partition, où chaque tuple représente une itération de la validation croisée.
      - X_train : Données d'entraînement pour la partition courante.
      - X_test : Données de test pour la partition courante.
      - Y_train : Classes d'entraînement pour la partition courante.
      - Y_test : Classes de test pour la partition courante.
    """
    # Fixer la graine pour reproduire le même résultat
    np.random.seed(seed)
    
    # Créer un tableau d'indices pour les données
    indices = np.arange(len(X))
    
    # Mélanger aléatoirement les indices pour créer une partition aléatoire
    np.random.shuffle(indices)

    # Appliquer les indices mélangés à X et Y
    X, Y = X[indices], Y[indices]

    # Calculer la taille de chaque partition
    fold_size = len(X) // k
    folds = []

    for i in range(k):
        # Définir les indices de début et de fin pour la partition de test
        start = i * fold_size
        end = (i + 1) * fold_size if i != k - 1 else len(X)  # Dernière partition peut être plus grande

        # Créer les ensembles de test et d'entraînement pour cette partition
        X_test, Y_test = X[start:end], Y[start:end]
        X_train = np.concatenate((X[:start], X[end:]), axis=0)
        Y_train = np.concatenate((Y[:start], Y[end:]), axis=0)

        # Ajouter la partition sous forme de tuple dans la liste
        folds.append((X_train, X_test, Y_train, Y_test))

    return folds

    

In [162]:
def knn_cross(X, Y):
    """
    Effectue une validation croisée pour déterminer la valeur optimale de k pour l'algorithme KNN.
    La fonction teste différentes valeurs de k (de 1 à 29, pas par pas de 2) et retourne la valeur de k
    qui minimise le taux d'erreur moyen sur les k-folds. Elle génère également un graphique de la courbe
    du taux d'erreur moyen en fonction de k.

    Paramètres :
    - X : Matrice des données (numpy array), de forme (n_samples, n_features).
    - Y : Vecteur des classes (numpy array), de forme (n_samples,).

    Retourne :
    - La valeur de k qui minimise le taux d'erreur moyen.
    - Le taux d'erreur minimum moyen correspondant à ce k.
    """
    # Diviser les données en 3 folds pour la validation croisée
    folds = k_fold_split(X, Y, k=3)
    
    # Tester les valeurs de k dans la plage de 1 à 29 (pas par pas de 2)
    k_values = np.arange(1, 30, 2)
    
    # Calculer le taux d'erreur moyen pour chaque valeur de k
    error_rates = np.array([np.mean([
        knn_error(k, X_train, X_test, Y_train, Y_test) 
        for (X_train, X_test, Y_train, Y_test) in folds
    ]) for k in k_values])
    
    # Tracer la courbe du taux d'erreur en fonction de k
    #plt.plot(k_values, error_rates)
    #plt.xlabel('Nombre de voisins (k)')
    #plt.ylabel('Taux d\'erreur moyen')
    #plt.title('Taux d\'erreur moyen en fonction de k pour KNN')
   # plt.show()
    
    # Retourner la valeur de k avec le taux d'erreur minimum
    optimal_k = k_values[np.argmin(error_rates)]
    min_error = np.min(error_rates)
    
    return optimal_k, min_error


> Maintenant, nous nous intéressons à trouver la meilleure combinaison de paramètres de manière locale (locale, car il est difficile (impossible) de tester toutes les combinaisons possibles).

In [163]:
def best_feature_combination(data, target_column, feature_columns, categorical_columns=[], max_features=3):
    """
    On va tester tous les combinaisons possibles de variables jusqu'a max_features et on va trouver celle avec 
    la taux minimal d'erreur plus petite
    Parametres:
        data (pd.DataFrame): Nos donnees.
        target_column (str): Le noms de columnes.
        feature_columns (list): List of potential feature names.
        categorical_columns (list): Sous ensemble des colonnes qui on besoin de utiliser label_encoder.
        max_features (int): Nombre maximal de variables qu'on va prendre au meme temps.

    Retour:
        (best_feature_tuple, best_score)
    """
    best_score = 100
    best_features = None

    # On va encoder les variables qui ne son pas de nombres
    Y, _= label_encoder(data[target_column].to_numpy())

    for r in range(4, max_features + 1):
        for combination in itertools.combinations(feature_columns, r):
            df_subset = data[list(combination)].copy()

            # On va encoder la classe
            for col in combination:
                if col in categorical_columns:
                    df_subset[col], _ = label_encoder(df_subset[col])

            #On normalise tous les variables
            X = normalise(df_subset.to_numpy())

            best_k, score = knn_cross(X, Y)

            if score < best_score:
                best_score = score
                best_features = combination

    return best_features, best_score, best_k

Ici, nous pouvons bien observer comment diviser les colonnes en fonction du type de données qu'elles contiennent.

> On trouve les meilleurs paramètres jusqu'à un maximum de 4 paramètres (cette partie prend beaucoup de temps à compiler).

> Attention a ne pas compiler cette partie chaue fois, je laisse le resultat en bas

In [164]:
"""feature_cols = ["quartier",	"site", "genre_arbre",	"situation",	"type_sol",	"surf_permeable",	"classe_age",
                "hauteur",	"classe_hauteur",	"diametre", "fissure_houppier",	"ecorce_incluse_houppier",	"bois_mort_houppier",
                "plaie_houppier",	"observation_houppier",	"esperance_maintien",	"contrainte", "Long",	"Lat"]
categorical_cols = ["quartier",	"site",	"genre_arbre",	"situation",	"type_sol",
                    "classe_age", "classe_hauteur","fissure_houppier", "ecorce_incluse_houppier",
                    "bois_mort_houppier", "plaie_houppier",	"observation_houppier", "contrainte"]
best_feats, best_score, best_k = best_feature_combination(
    data,
    target_column="classification_diagnostic",
    feature_columns=feature_cols,
    categorical_columns=categorical_cols,
    max_features=4 
)

print("✅ Best feature combo:", best_feats)
print("🎯 Best cross-val score:", round(best_score * 100, 2), "%")"""

'feature_cols = ["quartier",\t"site", "genre_arbre",\t"situation",\t"type_sol",\t"surf_permeable",\t"classe_age",\n                "hauteur",\t"classe_hauteur",\t"diametre", "fissure_houppier",\t"ecorce_incluse_houppier",\t"bois_mort_houppier",\n                "plaie_houppier",\t"observation_houppier",\t"esperance_maintien",\t"contrainte", "Long",\t"Lat"]\ncategorical_cols = ["quartier",\t"site",\t"genre_arbre",\t"situation",\t"type_sol",\n                    "classe_age", "classe_hauteur","fissure_houppier", "ecorce_incluse_houppier",\n                    "bois_mort_houppier", "plaie_houppier",\t"observation_houppier", "contrainte"]\nbest_feats, best_score, best_k = best_feature_combination(\n    data,\n    target_column="classification_diagnostic",\n    feature_columns=feature_cols,\n    categorical_columns=categorical_cols,\n    max_features=4 \n)\n\nprint("✅ Best feature combo:", best_feats)\nprint("🎯 Best cross-val score:", round(best_score * 100, 2), "%")'

✅ Best feature combo: ('genre_arbre', 'type_sol', 'hauteur', 'esperance_maintien')

🎯 Best cross-val score: 16.18 %

> Maintenant pour preparer une resultat avec knn et cette parametres

In [165]:

# Step 1: Select the best features
best_feats = ('genre_arbre', 'type_sol', 'hauteur', 'esperance_maintien')
df_subset = data[list(best_feats)].copy()
df_subset2 = prev_data[list(best_feats)].copy()

# Step 2: Encode categorical features
for col in best_feats:
    if col in categorical_cols:
        df_subset[col], _ = label_encoder(df_subset[col])
        df_subset2[col], _ = label_encoder(df_subset2[col])

# Step 3: Normalize
X_knn = normalise(df_subset.to_numpy())
X_pred = normalise(df_subset2.to_numpy())

# Step 4: Encode target
Y_train, mapping = label_encoder(data["classification_diagnostic"].to_numpy())

In [166]:
#Soumission 
k = best_k
# Prédictions sur prev_df
knn_predictions = knn(k,X_knn, X_pred, Y_train)
decode_labels(knn_predictions, mapping)

array(['C2', 'C2', 'C2', 'C1', 'C2', 'C1', 'C1', 'C1', 'C1', 'C2', 'C1',
       'C2', 'C2', 'C2', 'C1', 'C1', 'C2', 'C2', 'C2', 'C3', 'C2', 'C2',
       'C1', 'C1', 'C1', 'C2', 'C1', 'C2', 'C2', 'C1', 'C5', 'C2', 'C2',
       'C2', 'C2', 'C2', 'C2', 'C1', 'C2', 'C2', 'C1', 'C2', 'C1', 'C2',
       'C2', 'C1', 'C1', 'C2', 'C2', 'C2', 'C1', 'C1', 'C1', 'C2', 'C2',
       'C1', 'C2', 'C1', 'C2', 'C2', 'C1', 'C1', 'C1', 'C1', 'C2', 'C2',
       'C1', 'C2', 'C2', 'C1', 'C3', 'C1', 'C1', 'C2', 'C2', 'C1', 'C2',
       'C1', 'C1', 'C1', 'C1', 'C2', 'C2', 'C2', 'C1', 'C1', 'C2', 'C2',
       'C1', 'C1', 'C1', 'C1', 'C1', 'C1', 'C2', 'C2', 'C2', 'C1', 'C2',
       'C1', 'C2', 'C1', 'C2', 'C1', 'C2', 'C2', 'C1', 'C1', 'C2', 'C1',
       'C2', 'C2', 'C1', 'C1', 'C2', 'C2', 'C1', 'C2', 'C2', 'C2', 'C2',
       'C2', 'C2', 'C1', 'C1', 'C2', 'C2', 'C1', 'C2', 'C2', 'C2', 'C1',
       'C2', 'C2', 'C1', 'C1', 'C2'], dtype=object)

# RANDOM FOREST

# ARBRE DE DECISION

**Fonctions pour implementation**

In [167]:
def gini_impurity(labels : np.ndarray):
    """
    Calcul l'impureté de Gini pour un ensemble de prédictions.

    L'idée de l'impureté de Gini est la probabilité de se tromper dans une prédiction si on la prenanit au hasard au sein
    d'un échantillon donné.

    Entrées :
        labels : Un tableau de prédiction

    Sortie:
        La valeur d'impureté de Gini (float)
    """
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts/len(labels)
    return 1.-np.sum(probabilities**2)

In [168]:
def split_data( X_train, Y_train, feature_index, threshold):
    """
    Partitionne un ensemble d'entraînement (et de réponse) par rapport à une valeur de seuil.

    Entrée:
        X_train    : L'ensemble des données d'entraînement
        Y_train        : Les réponses correspondantes aux données d'entraînement
        feature_index : L'index du l'étiquette par rapport à laquelle on effectue le partitionnement
        threshold     : Le seuil définissant le partitionnement
    """
    left_mask  = X_train[:, feature_index] < threshold
    right_mask = ~left_mask
    return X_train[left_mask], Y_train[left_mask], X_train[right_mask], Y_train[right_mask]
    
def find_best_split(train_data : np.ndarray, labels : np.ndarray):
    """
    Trouve le meilleur partitionnement pour un ensemble de données d'entrée

    Entrées:
        train_data : Les données d'entrée
        labels     : Les prédictions correspondant à chaque entrée de l'entraînement

    Sortie:
        La meilleur étiquette (son index en fait) et la valeur de seuil pour son partitionnement
    """
    best_feature, best_threshold, best_gini = None, None, float('inf')
    for feature_index in range(train_data.shape[1]):
        # Récupère toutes les valeurs mesurées par l'étiquette courante (dans la boucle)
        values = np.unique(train_data[:, feature_index])
        # Recherche du seuil optimal pour cette étiquette :
        for threshold in values:
            _, left_labels, _, right_labels = split_data(train_data, labels, feature_index, threshold)
            if len(left_labels) == 0 or len(right_labels) == 0: continue
            # Calcul de l'indice de gini:
            gini = (len(left_labels)*gini_impurity(left_labels) + len(right_labels)*gini_impurity(right_labels)) / len(labels)
            # Plus l'indice de gini est petit, mieux c'est
            if gini < best_gini:
                best_feature, best_threshold, best_gini = feature_index, threshold, gini
                if best_gini == 0:
                    return best_feature, best_threshold
    return best_feature, best_threshold


In [169]:
def build_tree(X_train, Y_train, max_depth : int, min_samples_split : int, depth : int =  0):
    """
    Construit un arbre de décision récursivement

    Entrées :
        X_train       : Les données d'entrée
        Y_train            : Les prédictions correspondant à chaque entrée de l'entraînement
        max_depth         : La profondeur maximale de l'arbre
        min_samples_split : Le nombre d'échantillons minimal requis pour partitionner un noeud de l'arbre
        depth (optionnel) : La profondeur de l'arbre actuellement traitée (dans la récursion)

    Sortie:
        L'arbre de décision retourné sous la forme d'un dictionnaire
    """
    if depth == max_depth or len(Y_train) < min_samples_split or gini_impurity(Y_train) == 0:
        return {'prediction' : np.argmax(np.bincount(Y_train))}
    feature, threshold = find_best_split(X_train, Y_train)
    # Si pas trouvé de feature adéquat, on arrête la construction de la branche et
    # on recherche la valeur la plus probable comme réponse (sous forme d'indice) :
    if feature is None: return {"prediction" : np.argmax(np.bincount(Y_train))}
    left_train_data, left_labels, right_train_data, right_labels = split_data(X_train, Y_train, feature, threshold)
    return {
        "feature"   : feature,
        "threshold" : threshold,
        "samples": len(Y_train),  # Number of samples at this node
        "left"      : build_tree(left_train_data, left_labels, max_depth, min_samples_split, depth+1),
        "right"     : build_tree(right_train_data, right_labels, max_depth, min_samples_split, depth+1)
    }

In [170]:
def predict(tree : dict, data_point : np.ndarray):
    """
    Fait une prédiction pour un simple échantillon utilisant l'arbre de décision

    Entrées:
        tree       : L'arbre de décision représentée comme un dictionnaire
        data_point : L'échantillon d'entrée

    Sortie:
        La valeur prédite pour l'étiquette
    """
    data_point = np.asarray(data_point).flatten()
    # Base case: return prediction if at a leaf node
    if 'prediction' in tree:
        return tree['prediction']
    
    # Get feature and threshold, ensuring they exist
    feature = tree.get('feature')
    threshold = tree.get('threshold')

    if feature is None or threshold is None or feature >= len(data_point):
        raise ValueError("Invalid tree structure or feature index out of bounds.")

    # Recurse down the correct branch
    if data_point[feature] < threshold:
        return predict(tree['left'], data_point)
    else:
        return predict(tree['right'], data_point)

In [171]:
def predict_batch(tree, X_test):
    """
    Predicts labels for multiple samples using a trained decision tree.

    Inputs:
        tree   : The decision tree model (a dictionary)
        X_test : A NumPy array of test data (each row is a sample)

    Output:
        A NumPy array with predicted labels
    """
    return np.array([predict(tree, x) for x in X_test])

# RANDOM FOREST IMPLEMENTATION

In [172]:
def classe_majoritaire(l):
    counts = np.bincount(l)
    return np.argmax(counts)

In [173]:
def choose_feature(nb_feature, train_data, test_data):
    """
    Sélectionne un sous-ensemble de caractéristiques pour chaque arbre de décision.
    
    Entrées:
        nb_feature : Nombre de caractéristiques à sélectionner pour chaque arbre
        train_data : Données d'entraînement (matrice NumPy)
        test_data  : Données de test (matrice NumPy)
        
    Sortie:
        train      : Données d'entraînement pour le sous-ensemble sélectionné
        test       : Données de test pour le sous-ensemble sélectionné
        selected_cols : Liste des noms des colonnes sélectionnées (ou indices des colonnes)
    """
    # Obtenons le nombre de colonnes/features
    _, cols = train_data.shape
    
    # Sélectionner un sous-ensemble de features (colonnes) au hasard
    selected_indices = random.sample(range(cols), nb_feature)
    
    # Extraire les colonnes correspondantes des données d'entraînement et de test
    train = train_data[:, selected_indices]
    test = test_data[:, selected_indices]
    
    # Les colonnes sélectionnées (noms des features ou indices des colonnes)
    selected_cols = selected_indices
    
    return train, test, selected_cols

In [174]:
def random_forest(nb_tree, nb_feature, train_data, train_labels, test_data, max_depth, min_sample):
    """
    Construire une forêt d'arbres de décision et faire une prédiction pour les données de test.

    Entrées:
        nb_tree        : Nombre d'arbres dans la forêt
        nb_feature     : Nombre de caractéristiques à sélectionner pour chaque arbre
        train_data     : Les données d'entraînement
        train_labels   : Les étiquettes associées aux données d'entraînement
        test_data      : Les données de test
        max_depth      : La profondeur maximale des arbres
        min_sample     : Le nombre minimal d'échantillons pour une division

    Sortie:
        Une matrice de prédictions, une pour chaque arbre, en prenant la classe majoritaire.
    """
    predictions = []  # Liste pour stocker les prédictions de chaque arbre
    
    # Construire les arbres un par un
    for k in range(nb_tree):
        # Sélectionner un sous-ensemble de caractéristiques aléatoires pour chaque arbre
        train, test, _ = choose_feature(nb_feature, train_data, test_data)
        
        # Construire un arbre de décision
        tree = build_tree(train, train_labels, max_depth, min_sample, 0)
        
        # Effectuer des prédictions pour le test_data
        prediction = [predict(tree, test[i]) for i in range(len(test_data))]
        
        # Ajouter les prédictions de cet arbre à la liste
        predictions.append(prediction)
    
    # Convertir la liste de prédictions en un tableau NumPy pour le traitement
    predictions = np.array(predictions).T
    
    # Prédiction finale: classe majoritaire parmi toutes les prédictions des arbres
    final_prediction = [classe_majoritaire(p) for p in predictions]
    
    return np.array(final_prediction)

# Maintenant on fait un grid search pour trouver les meilleurs hyperparametres

In [175]:
def cross_validate_random_forest(X, Y, k=10, nb_tree=100, nb_feature=10, max_depth=20, min_sample=2):
    folds = k_fold_split(X, Y, k=k)  # Split data into k folds
    scores = []

    # Loop through each fold
    for i, (X_train, X_test, Y_train, Y_test) in enumerate(folds):
        # Train and predict using the Random Forest model
        predictions = random_forest(nb_tree, nb_feature, X_train, Y_train, X_test, max_depth, min_sample)
        
        # Calculate accuracy
        accuracy = np.mean(predictions == Y_test)
        scores.append(accuracy)
        
        # Print fold-specific results
        print(f"Fold {i+1}: Accuracy = {accuracy:.4f}")

    # Calculate the mean accuracy across all folds
    mean_score = np.mean(scores)
    print(f"\nMean accuracy across {k} folds: {mean_score:.4f}")

    return mean_score

from itertools import product

def grid_search_random_forest(X, Y, k=10, param_grid=None):
    """
    Effectue une recherche par grille pour optimiser les hyperparamètres d'un Random Forest
    Parametres:
    - X: Matrice des caractéristiques (numpy array)
    - Y: Vecteur des cibles (numpy array)
    - k: Nombre de plis pour la validation croisée
    - param_grid: Dictionnaire de la grille des hyperparamètres à tester
    Retourne le meilleur ensemble d'hyperparamètres
    """
    if param_grid is None:
        param_grid = {
            'nb_tree': [50, 100, 200],
            'nb_feature': [5, 10, 15],
            'max_depth': [10, 20, 30],
            'min_sample': [1, 2, 5]
        }

    # Générer toutes les combinaisons possibles des hyperparamètres
    param_combinations = list(product(*param_grid.values()))
    
    best_score = -float('inf')
    best_params = None

    # Essayer chaque combinaison d'hyperparamètres
    for combination in param_combinations:
        # Mapper les valeurs aux noms des hyperparamètres
        params = dict(zip(param_grid.keys(), combination))
        
        print(f"Testing hyperparameters: {params}")
        
        # Effectuer la validation croisée avec cette combinaison d'hyperparamètres
        mean_score = cross_validate_random_forest(
            X, Y, k=k, 
            nb_tree=params['nb_tree'], 
            nb_feature=params['nb_feature'], 
            max_depth=params['max_depth'], 
            min_sample=params['min_sample']
        )
        
        # Si ce score est meilleur que le précédent, on met à jour le meilleur score
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"\nBest hyperparameters found: {best_params}")
    return best_params, best_score

In [176]:
def cross_validate_random_forest(X, Y, k=10, nb_tree=100, nb_feature=10, max_depth=20, min_sample=2):
    folds = k_fold_split(X, Y, k=k)  # Split data into k folds
    scores = []

    # Loop through each fold
    for i, (X_train, X_test, Y_train, Y_test) in enumerate(folds):
        # Train and predict using the Random Forest model
        predictions = random_forest(nb_tree, nb_feature, X_train, Y_train, X_test, max_depth, min_sample)
        
        # Calculate accuracy
        accuracy = np.mean(predictions == Y_test)
        scores.append(accuracy)
        
        # Print fold-specific results
        print(f"Fold {i+1}: Accuracy = {accuracy:.4f}")

    # Calculate the mean accuracy across all folds
    mean_score = np.mean(scores)
    print(f"\nMean accuracy across {k} folds: {mean_score:.4f}")

    return mean_score

from itertools import product

def grid_search_random_forest(X, Y, k=10, param_grid=None):
    """
    Effectue une recherche par grille pour optimiser les hyperparamètres d'un Random Forest
    Parametres:
    - X: Matrice des caractéristiques (numpy array)
    - Y: Vecteur des cibles (numpy array)
    - k: Nombre de plis pour la validation croisée
    - param_grid: Dictionnaire de la grille des hyperparamètres à tester
    Retourne le meilleur ensemble d'hyperparamètres
    """
    if param_grid is None:
        param_grid = {
            'nb_tree': [50, 100, 200],
            'nb_feature': [5, 10, 15],
            'max_depth': [10, 20, 30],
            'min_sample': [1, 2, 5]
        }

    # Générer toutes les combinaisons possibles des hyperparamètres
    param_combinations = list(product(*param_grid.values()))
    
    best_score = -float('inf')
    best_params = None

    # Essayer chaque combinaison d'hyperparamètres
    for combination in param_combinations:
        # Mapper les valeurs aux noms des hyperparamètres
        params = dict(zip(param_grid.keys(), combination))
        
        print(f"Testing hyperparameters: {params}")
        
        # Effectuer la validation croisée avec cette combinaison d'hyperparamètres
        mean_score = cross_validate_random_forest(
            X, Y, k=k, 
            nb_tree=params['nb_tree'], 
            nb_feature=params['nb_feature'], 
            max_depth=params['max_depth'], 
            min_sample=params['min_sample']
        )
        
        # Si ce score est meilleur que le précédent, on met à jour le meilleur score
        if mean_score > best_score:
            best_score = mean_score
            best_params = params

    print(f"\nBest hyperparameters found: {best_params}")
    return best_params, best_score

> Un autre code qu'il faut pas compiler, je donne le resultat de parametres

In [177]:
"""# Definisons le grid de parametres
param_grid = {
    'nb_tree': [300, 400, 500],           # Number of trees
    'nb_feature': [8, 9, 10],           # Number of features to choose per tree
    'max_depth': [12, 20, 30],           # Maximum depth of each tree
    'min_sample': [8, 9, 10]              # Minimum number of samples per leaf
}

# Perform the grid search
best_params, best_score = grid_search_random_forest(X, Y, k=3, param_grid=param_grid)

print(f"Best parameters: {best_params}")
print(f"Best accuracy: {best_score:.4f}")"""

'# Definisons le grid de parametres\nparam_grid = {\n    \'nb_tree\': [300, 400, 500],           # Number of trees\n    \'nb_feature\': [8, 9, 10],           # Number of features to choose per tree\n    \'max_depth\': [12, 20, 30],           # Maximum depth of each tree\n    \'min_sample\': [8, 9, 10]              # Minimum number of samples per leaf\n}\n\n# Perform the grid search\nbest_params, best_score = grid_search_random_forest(X, Y, k=3, param_grid=param_grid)\n\nprint(f"Best parameters: {best_params}")\nprint(f"Best accuracy: {best_score:.4f}")'

    nb_tree=300, 
    nb_feature=10, 
    train_data= X, 
    train_labels= Y, 
    test_data= X_prev , 
    max_depth=12, 
    min_sample=9

In [178]:
feature_cols = ["quartier", "site", "genre_arbre", "situation", "type_sol", "surf_permeable", "classe_age",
                "hauteur", "classe_hauteur", "diametre", "fissure_houppier", "ecorce_incluse_houppier", "bois_mort_houppier",
                "plaie_houppier", "observation_houppier", "esperance_maintien", "contrainte", "Long", "Lat"]
categorical_cols = ["quartier", "site", "genre_arbre", "situation", "type_sol",
                    "classe_age", "classe_hauteur", "fissure_houppier", "ecorce_incluse_houppier",
                    "bois_mort_houppier", "plaie_houppier", "observation_houppier", "contrainte"]

In [179]:
# Colonne cible
target_column = "Usage"

# Appliquer l'encodage des étiquettes aux caractéristiques catégorielles et stocker les mappings
for col in categorical_cols:
    prev_data[col], _ = label_encoder(prev_data[col])

feature_cols = categorical_cols + [col for col in prev_data.columns if col not in categorical_cols + [target_column]]

# Convert to NumPy after encoding
X_rf_prev = prev_data[feature_cols].to_numpy()

In [180]:
# Colonne cible (nécessite un encodage)
target_column = "classification_diagnostic"

# Appliquer l'encodage des étiquettes aux caractéristiques catégorielles et stocker les mappings
for col in categorical_cols:
    data[col], _ = label_encoder(data[col])

feature_cols = categorical_cols + [col for col in data.columns if col not in categorical_cols + [target_column]]

# Convertir en NumPy après encodage
X_rf = data[feature_cols].to_numpy()

In [181]:
# Encoder la variable cible (en conservant le mapping)
Y, mapping = label_encoder(data[target_column].to_numpy())

In [ ]:
rf_predictions = random_forest(
    nb_tree=300, 
    nb_feature=10, 
    train_data= X_rf, 
    train_labels= Y, 
    test_data= X_rf_prev , 
    max_depth=12, 
    min_sample=9
)

decode_labels(rf_predictions, mapping)

# MODELE ENSEMBLISTE

In [ ]:
#On va continuer a diviser nos modeles et en utilisant la validation croisee

> Now that we have both of our predictions we are going to give a weight to each one and select the one with the best output

In [ ]:
def ensemble_knn_rf(X_train_rf, X_test_rf, X_train_knn, X_test_knn, Y_train, rf_weight, knn_weight):
    """
    Combine les prédictions de KNN et de Random Forest en utilisant une moyenne pondérée.
    
    Arguments:
    - X_train_rf : Données d'entraînement pour Random Forest (tableau numpy 2D).
    - X_test_rf : Données de test pour Random Forest (tableau numpy 2D).
    - X_train_knn : Données d'entraînement pour KNN (tableau numpy 2D).
    - X_test_knn : Données de test pour KNN (tableau numpy 2D).
    - Y_train : Étiquettes d'entraînement (tableau numpy 1D).
    - rf_weight : Poids des prédictions de Random Forest (float).
    - knn_weight : Poids des prédictions de KNN (float).
    
    Retourne:
    - ensemble_predictions : Prédictions combinées (tableau numpy 1D).
    """
    # Prédictions de Random Forest
    rf_predictions = random_forest(
        nb_tree=300, 
        nb_feature=10, 
        train_data=X_train_rf, 
        train_labels=Y_train, 
        test_data=X_test_rf, 
        max_depth=12, 
        min_sample=9
    )

    # Prédictions de KNN
    knn_predictions = knn(5, X_train_knn, X_test_knn, Y_train)

    # Prédictions de l'ensemble pondéré
    ensemble_predictions = (knn_weight * knn_predictions) + (rf_weight * rf_predictions)
    ensemble_predictions = np.round(ensemble_predictions).astype(int)
    
    return ensemble_predictions

def cross_ensemble(X_rf, X_knn, Y, rf_weight, knn_weight, k=10):
    """
    Effectuer la validation croisée à k plis pour un ensemble des modèles KNN et Random Forest.
    
    Arguments:
    - X_rf : Matrice de caractéristiques pour Random Forest (tableau numpy).
    - X_knn : Matrice de caractéristiques pour KNN (tableau numpy).
    - Y : Étiquettes (tableau numpy 1D).
    - rf_weight : Poids des prédictions de Random Forest (float).
    - knn_weight : Poids des prédictions de KNN (float).
    - k : Nombre de plis pour la validation croisée (entier).
    
    Retourne:
    - mean_score : Précision moyenne à travers tous les plis (float).
    """
    folds_rf = k_fold_split(X_rf, Y, k=k, seed=42)
    folds_knn = k_fold_split(X_knn, Y, k=k, seed=42)

    scores = []  # Liste pour stocker les précisions de chaque pli

    # Boucle à travers chaque pli
    for i, ((X_train_rf, X_test_rf, Y_train, Y_test), (X_train_knn, X_test_knn, _, _)) in enumerate(zip(folds_rf, folds_knn)):
        # Faire des prédictions en utilisant l'ensemble des modèles
        ensemble_predictions = ensemble_knn_rf(X_train_rf, X_test_rf, X_train_knn, X_test_knn, Y_train, rf_weight, knn_weight)
        
        # Calculer la précision
        accuracy = np.mean(ensemble_predictions == Y_test)
        scores.append(accuracy)
        
        # Afficher les résultats spécifiques à chaque pli
        print(f"Plis {i+1} : Précision = {accuracy:.4f}")

    # Calculer la précision moyenne à travers tous les plis
    mean_score = np.mean(scores)
    print(f"\nPrécision moyenne à travers {k} plis : {mean_score:.4f}")

    return mean_score

In [ ]:
#voting_preds

In [ ]:
#decode_labels(voting_preds, mapping)

In [ ]:
"""# Réassocier correctement les ID_ARBRE dans la soumission
df_submission = pd.DataFrame({
    'ID_ARBRE': prev_data.index,  # Utiliser l'index correctement
    'classification_diagnostic': decode_labels(voting_preds, mapping)
})

# Sauvegarde du fichier corrigé
submission_path_fixed = "/kaggle/working/submission.csv"


df_submission.to_csv(submission_path_fixed, index=False)

print("Fichier 'soumission_fixed.csv' créé avec succès !")"""

In [ ]:
"""
df['classification_diagnostic'] = ''
df['classification_diagnostic'] = decode_labels(voting_preds, mapping) # np.where(df['INDEX']== 0, labels[0], np.where(df['INDEX']== 1, labels[1], labels[2]))
df.drop(columns=['INDEX'],inplace=True)
df['ID_ARBRE'] = prev_data.index
df = df.set_index('ID_ARBRE')
print(f"df 3 : {df}")
df.to_csv("submission.csv")"""